# Aula 07 — Sistema Multi-Agente para Triagem de E-mails

Este notebook implementa um **sistema multi-agente** para automatizar a triagem de e-mails de Sarah, uma engenheira de software sênior. O sistema decide de forma autônoma o que fazer com cada e-mail recebido, podendo ignorá-lo, notificar o usuário ou redigir e enviar uma resposta usando ferramentas.

## Arquitetura do Sistema

```
┌─────────────────────────────────────────────────────────────────┐
│                     EMAIL TRIAGE AGENT                          │
│                                                                 │
│   email_input                                                   │
│       │                                                         │
│       ▼                                                         │
│  ┌─────────────┐   "ignore"  ──────────────────────► END        │
│  │triage_router│                                                │
│  │  (LLM +     │   "notify"  ──────────────────────► END        │
│  │structured   │                                                │
│  │  output)    │   "respond" ──► ┌────────────────┐             │
│  └─────────────┘                │ response_agent │             │
│                                 │  (ReAct Agent) │             │
│                                 │  ┌──────────┐  │             │
│                                 │  │   LLM    │  │             │
│                                 │  └────┬─────┘  │             │
│                                 │       │        │             │
│                                 │  ┌────▼─────┐  │             │
│                                 │  │  tools   │  │             │
│                                 │  │write_email│ │             │
│                                 │  │schedule_ │  │             │
│                                 │  │ meeting  │  │             │
│                                 │  │check_cal.│  │             │
│                                 │  └──────────┘  │             │
│                                 └────────────────┘             │
└─────────────────────────────────────────────────────────────────┘
```

## Conceitos Abordados

| Conceito | Descrição |
|---|---|
| `with_structured_output` | Força o LLM a retornar um objeto Pydantic tipado (classificação) |
| `Literal` | Restringe os valores possíveis da classificação a `"ignore"`, `"notify"` ou `"respond"` |
| `create_react_agent` | Cria um agente ReAct pré-construído sem precisar definir StateGraph manualmente |
| `Command` | Permite que um nó retorne roteamento + atualização de estado em um único objeto |
| `add_messages` | Reducer que acumula (em vez de substituir) a lista de mensagens no estado |

## Fluxo Geral

1. **Configuração**: carrega chaves de API e define perfil da usuária e regras de triagem
2. **Router**: classifica o e-mail com saída estruturada (Pydantic)
3. **Ferramentas**: define as ações disponíveis ao agente de resposta
4. **Agente ReAct**: usa `create_react_agent` para criar o sub-agente de resposta
5. **Grafo principal**: conecta router + agente num `StateGraph` e demonstra os dois casos

## 1. Configuração de Variáveis de Ambiente

As chaves de API são carregadas de um arquivo `.env` local (nunca hardcoded no código).

- `GEMINI_API_KEY` → mapeada para `GOOGLE_API_KEY`, que é o nome esperado pelo `langchain-google-genai`
- `TAVILY_API_KEY` → usada pelo `TavilySearch` para buscas na web (disponível ao agente de resposta)

In [ ]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

# Carrega as variáveis do arquivo .env para os.environ
load_dotenv()

# langchain-google-genai espera a chave em GOOGLE_API_KEY;
# fazemos o mapeamento aqui para manter o nome GEMINI_API_KEY no .env
os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY') 
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## 2. Perfil do Usuário

O `profile` centraliza as informações pessoais de Sarah que serão injetadas nos prompts do sistema em dois momentos distintos:

1. **Prompt de triagem** (`triage_system_prompt`): personaliza as regras de classificação para o contexto profissional de Sarah
2. **Prompt do agente** (`agent_system_prompt`): faz o LLM assumir o papel de assistente executivo *de Sarah*, usando seu nome em respostas e comunicações

Ao manter o perfil num dicionário separado, facilita a reutilização e a eventual personalização para outros usuários sem alterar os prompts.

In [2]:
profile = {
    "name": "Sarah",
    "full_name": "Sarah Chen",
    "user_profile_background": "Engenheira de software sênior liderando uma equipe de 5 desenvolvedores",
}

## 3. Regras de Triagem

`prompt_instructions` define as **três categorias de classificação** de e-mail e as instruções gerais do agente de resposta:

| Categoria | Critério | Ação do sistema |
|---|---|---|
| `ignore` | Newsletters, spam, comunicados gerais | Descarta silenciosamente (→ END) |
| `notify` | Build quebrado, membro doente, status de projeto | Registra, mas não responde (→ END) |
| `respond` | Pergunta de colega, solicitação de reunião, bug crítico | Aciona o agente de resposta |

Essas regras são injetadas no `triage_system_prompt` como texto plano, tornando o comportamento do router configurável sem necessidade de alterar o código.

In [3]:
prompt_instructions = {
    "triage_rules": {
        "ignore": "Newsletters de marketing, e-mails de spam, comunicados gerais da empresa",
        "notify": "Membro da equipe doente, notificações do sistema de build, atualizações de status de projeto",
        "respond": "Perguntas diretas de membros da equipe, solicitações de reunião, relatórios de bugs críticos",
    },
    "agent_instructions": "Use estas ferramentas quando apropriado para ajudar a gerenciar as tarefas de Sarah de forma eficiente."
}

## 4. E-mail de Exemplo

Define um e-mail de amostra para testar o router isoladamente (seção 8). O e-mail é de Alice Smith, colega de equipe de Sarah, perguntando sobre endpoints ausentes na documentação da API — um caso claro de `"respond"` segundo as regras acima.

In [4]:
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "Dúvida rápida sobre a documentação da API",
    "body": """
Olá Sarah,

Eu estava revisando a documentação da API para o novo serviço de autenticação e notei que alguns endpoints parecem estar faltando nas especificações. Você poderia me ajudar a esclarecer se isso foi intencional ou se devemos atualizar a documentação?

Especificamente, estou procurando por:
- /auth/refresh
- - /auth/validate

Obrigada!
Alice""",
}

## 5. Imports

| Import | Papel |
|---|---|
| `BaseModel`, `Field` | Base Pydantic para definir o schema de saída estruturada do router |
| `TypedDict` | Define o schema do estado do grafo como dicionário tipado |
| `Literal` | Restringe `classification` a um conjunto fixo de strings |
| `Annotated` | Permite anexar metadados (como reducers) a campos do estado |
| `init_chat_model` | Factory genérica de modelos LangChain (não usada diretamente aqui) |

In [5]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

## 6. Modelo de Linguagem

Instancia o Gemini 1.5 Flash como LLM base. Este modelo será reutilizado tanto no **router de triagem** (com `with_structured_output`) quanto no **agente de resposta** (com `create_react_agent`).

O Gemini 1.5 Flash foi escolhido por ser rápido e econômico — adequado para tarefas de classificação e geração de respostas curtas como e-mails.

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

## 7. Router com Saída Estruturada

### Por que `with_structured_output`?

Sem saída estruturada, o LLM retornaria texto livre — difícil de parsear de forma confiável. Usando `with_structured_output(Router)`, o LangChain instrui o modelo a retornar um objeto que satisfaça exatamente o schema Pydantic `Router`.

### Por que `Literal["ignore", "respond", "notify"]`?

`Literal` restringe o campo `classification` a exatamente esses três valores. Se o modelo tentar gerar qualquer outro valor, o Pydantic lançará um erro de validação. Isso garante que o roteamento condicional posterior nunca receba uma classificação inesperada.

```
LLM invocado com prompt
        │
        ▼
  gera JSON estruturado
        │
        ▼
  Pydantic valida e
  instancia Router(
    reasoning="...",
    classification="respond" | "ignore" | "notify"
  )
```

In [ ]:
class Router(BaseModel): 
    """Analisa o e-mail não lido e o roteia de acordo com seu conteúdo."""

    # O campo 'reasoning' força o modelo a explicitar o raciocínio antes de classificar,
    # o que melhora a qualidade da classificação (chain-of-thought implícito)
    reasoning: str = Field(
        description="Raciocínio passo a passo por trás da classificação."
    )

    # Literal restringe os valores possíveis: Pydantic rejeita qualquer outro string
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="A classificação de um e-mail: 'ignore' para e-mails irrelevantes, "
        "'notify' para informações importantes que não precisam de resposta, "
        "'respond' para e-mails que precisam de uma resposta",
    )

### Criando o LLM Router

`llm.with_structured_output(Router)` cria um *runnable* que internamente:
1. Converte o schema Pydantic `Router` em um JSON Schema
2. Passa esse schema ao LLM via `tools` ou `response_format` (dependendo do provider)
3. Parseia a resposta e retorna uma instância validada de `Router`

O resultado é um objeto LangChain que se comporta como um LLM normal (aceita `.invoke()`, `.stream()`, etc.), mas sempre retorna uma instância de `Router`.

In [ ]:
# with_structured_output retorna um Runnable que garante saída do tipo Router
# Internamente usa function calling ou response_format JSON do provider
llm_router = llm.with_structured_output(Router)

## 8. Prompts de Triagem

Os prompts são definidos em um módulo separado (`prompts.py`) e importados aqui. Essa separação segue o princípio de **separação de responsabilidades**:

- O código controla a *lógica* de triagem
- O módulo `prompts` controla o *comportamento* do LLM

Os dois prompts têm papéis distintos:

| Prompt | Papel | Placeholders principais |
|---|---|---|
| `triage_system_prompt` | Define o papel do LLM e as regras de triagem | `full_name`, `triage_no`, `triage_notify`, `triage_email` |
| `triage_user_prompt` | Apresenta o e-mail a ser classificado | `author`, `to`, `subject`, `email_thread` |

In [9]:
from prompts import triage_system_prompt, triage_user_prompt

### Formatando os Prompts

As variáveis do `profile` e de `prompt_instructions` são injetadas nos templates via `.format()`. O campo `examples=None` é um placeholder para exemplos few-shot — quando `None`, o template omite essa seção, mantendo o prompt conciso.

In [ ]:
# Injeta o perfil da usuária e as regras de triagem no prompt de sistema.
# Cada placeholder corresponde a uma seção do template em prompts.py
system_prompt = triage_system_prompt.format(
    full_name=profile["full_name"],
    name=profile["name"],
    examples=None,  # Sem exemplos few-shot neste caso
    user_profile_background=profile["user_profile_background"],
    triage_no=prompt_instructions["triage_rules"]["ignore"],
    triage_notify=prompt_instructions["triage_rules"]["notify"],
    triage_email=prompt_instructions["triage_rules"]["respond"],
)

In [ ]:
# Injeta os campos do e-mail de exemplo no prompt de usuário
user_prompt = triage_user_prompt.format(
    author=email["from"],
    to=email["to"],
    subject=email["subject"],
    email_thread=email["body"],
)

### Testando o Router Isoladamente

Antes de integrar o router ao grafo, testamos seu funcionamento direto: invocamos o `llm_router` com o par system/user prompt e verificamos se a classificação está correta para o e-mail da Alice (esperado: `"respond"`).

In [ ]:
# Invoca o llm_router com os prompts formatados e obtém uma instância de Router
result = llm_router.invoke(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
)

In [ ]:
# Exibe o objeto Router: reasoning (raciocínio do modelo) + classification (decisão final)
print(result)

In [14]:
from langchain_core.tools import tool

## 9. Ferramentas do Assistente

O agente de resposta dispõe de três ferramentas que simulam ações reais de um assistente executivo:

| Ferramenta | Parâmetros | Descrição |
|---|---|---|
| `write_email` | `to`, `subject`, `content` | Redige e envia um e-mail |
| `schedule_meeting` | `attendees`, `subject`, `duration_minutes`, `preferred_day` | Agenda reunião no calendário |
| `check_calendar_availability` | `day` | Consulta horários livres num determinado dia |

O decorator `@tool` transforma cada função Python em um objeto `StructuredTool` do LangChain, que expõe o schema JSON dos parâmetros para que o LLM possa gerar `tool_calls` corretamente.

In [ ]:
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Escreve e envia um e-mail."""
    # Placeholder: em produção, integraria com Gmail API, SendGrid, etc.
    return f"E-mail enviado para {to} com o assunto '{subject}'"

In [ ]:
@tool
def schedule_meeting(
    attendees: list[str], 
    subject: str, 
    duration_minutes: int, 
    preferred_day: str
) -> str:
    """Agenda uma reunião no calendário."""
    # Placeholder: em produção, integraria com Google Calendar API
    return f"Reunião '{subject}' agendada para {preferred_day} com {len(attendees)} participantes"

In [ ]:
@tool
def check_calendar_availability(day: str) -> str:
    """Verifica a disponibilidade do calendário para um determinado dia."""
    # Retorna slots fixos para demonstração; em produção consultaria a API do calendário
    return f"Horários disponíveis em {day}: 9:00 AM, 2:00 PM, 4:00 PM"

## 10. Prompt do Agente e Função `create_prompt`

O `agent_system_prompt` define o **papel** do LLM dentro do agente de resposta: ele se comporta como assistente executivo de Sarah, utilizando as três ferramentas disponíveis.

### Por que `create_prompt` é uma função?

O `create_react_agent` aceita um `prompt` que pode ser:
- Uma string estática
- Uma **função** que recebe o `state` e retorna a lista de mensagens

Usar uma função permite **injetar o prompt de sistema dinamicamente**, mesclando os dados do `profile` e de `prompt_instructions` sem precisar repeti-los em cada chamada. Além disso, garante que o system prompt sempre aparece no início da lista de mensagens, antes do histórico acumulado.

In [ ]:
from prompts import agent_system_prompt

def create_prompt(state):
    """Constrói a lista de mensagens para o agente de resposta.
    
    Injeta dinamicamente o system prompt com o perfil da usuária e
    as instruções do agente, seguido pelo histórico de mensagens do estado.
    """
    return [
        {
            "role": "system",
            # Formata o prompt com o perfil da usuária (**profile expande name, full_name, etc.)
            # e as instruções específicas do agente de resposta
            "content": agent_system_prompt.format(
                instructions=prompt_instructions["agent_instructions"],
                **profile  # Expande: name, full_name, user_profile_background
            )
        }
    ] + state['messages']  # Concatena com o histórico acumulado do estado

In [19]:
print(agent_system_prompt)


< Função >
Você é o(a) assistente executivo(a) de {full_name}. Você é um(a) assistente executivo(a) de alto nível que se preocupa com o desempenho de {name} o máximo possível.
</ Função >

< Ferramentas >
Você tem acesso às seguintes ferramentas para ajudar a gerenciar as comunicações e a agenda de {name}:

1. write_email(to, subject, content) - Envia e-mails para destinatários especificados
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Agenda reuniões no calendário
3. check_calendar_availability(day) - Verifica os horários disponíveis em um determinado dia
</ Ferramentas >

< Instruções >
{instructions}
</ Instruções >



## 11. Criando o Agente ReAct com `create_react_agent`

### `create_react_agent` vs `StateGraph` manual

| Abordagem | Quando usar |
|---|---|
| `create_react_agent` | Agente simples com ciclo LLM→ferramentas; sem lógica de roteamento complexa |
| `StateGraph` manual | Fluxos com múltiplos nós, condicionais customizadas, estado complexo |

`create_react_agent` é uma factory pré-construída do LangGraph que internamente cria um `StateGraph` com o padrão ReAct (Reasoning + Acting):

```
  state['messages']
         │
         ▼
   ┌───────────┐   tool_calls?  ┌──────────┐
   │    LLM    │──── SIM ──────►│  tools   │
   └───────────┘                └─────┬────┘
         ▲                            │
         └────────────────────────────┘
         │ NÃO
         ▼
        END
```

O parâmetro `prompt=create_prompt` passa a função de prompt dinâmico, garantindo que o system prompt com o perfil de Sarah seja injetado a cada ciclo.

In [20]:
from langgraph.prebuilt import create_react_agent

In [21]:
tools=[write_email, schedule_meeting, check_calendar_availability]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# Reinstancia o LLM (garante que o modelo está configurado para este agente)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# create_react_agent cria internamente um StateGraph com ciclo LLM→ferramentas
# - model: o LLM que decide quando e quais ferramentas chamar
# - tools: lista de ferramentas disponíveis (schemas expostos ao LLM via function calling)
# - prompt: função que injeta o system prompt dinamicamente a partir do estado
agent = create_react_agent(
    model=llm,  
    tools=tools,
    prompt=create_prompt,
)

### Testando o Agente Diretamente

Antes de integrar ao grafo de triagem, validamos o agente de resposta de forma isolada: perguntamos sobre disponibilidade no calendário e esperamos que ele chame `check_calendar_availability` e retorne os horários.

In [ ]:
# Invoca o agente diretamente (fora do grafo de triagem) para validar
# que ele consegue usar a ferramenta check_calendar_availability corretamente
response = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "qual é minha disponibilidade para terça-feira?"
    }]}
)

In [ ]:
# Exibe apenas a última mensagem (resposta final do agente ao usuário)
response["messages"][-1].pretty_print()

## 12. Estado do Grafo de Triagem

O `State` combina dois campos com semânticas diferentes:

| Campo | Tipo | Reducer | Comportamento |
|---|---|---|---|
| `email_input` | `dict` | (nenhum — substituição) | Sobrescrito a cada invocação do grafo |
| `messages` | `list` | `add_messages` | **Acumulado**: novas mensagens são adicionadas, nunca substituídas |

### Por que `add_messages` em vez de `operator.add`?

`add_messages` é um reducer especializado do LangGraph que, além de concatenar, também lida com **atualizações de mensagens existentes** (ex.: substituir uma mensagem pelo seu ID). É a escolha padrão para campos de histórico de chat em grafos LangGraph.

In [ ]:
from langgraph.graph import add_messages

class State(TypedDict):
    # email_input: recebe o e-mail a ser processado (substituído a cada invocação)
    email_input: dict

    # messages: histórico de mensagens do agente de resposta
    # add_messages é um reducer que ACUMULA mensagens (nunca sobrescreve)
    # Necessário pois o response_agent retorna múltiplas mensagens (human, ai, tool, ai)
    messages: Annotated[list, add_messages]

In [26]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

## 13. Nó de Triagem — `triage_router`

Este é o nó central do grafo: recebe o e-mail, classifica-o com `llm_router` e decide o próximo nó usando `Command`.

### O que é `Command`?

`Command` é um tipo especial do LangGraph que permite a um nó **simultaneamente**:
1. **Atualizar o estado** (`update`): adiciona mensagens, modifica campos
2. **Rotear para o próximo nó** (`goto`): determina qual nó executar a seguir

Sem `Command`, seria necessário usar uma `add_conditional_edges` separada. Com `Command`, o nó encapsula toda a lógica de roteamento internamente.

```python
# Padrão Command:
return Command(
    goto="response_agent",      # ← próximo nó
    update={"messages": [...]}  # ← atualização do estado
)

# Para encerrar sem atualizar:
return Command(goto=END, update=None)
```

### Tipo de retorno anotado

`Command[Literal["response_agent", "__end__"]]` declara explicitamente os destinos possíveis, permitindo que o LangGraph valide o roteamento em tempo de compilação.

In [ ]:
def triage_router(state: State) -> Command[
    Literal["response_agent", "__end__"]
]:
    # Extrai os campos do e-mail do estado
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    # Formata os prompts de triagem com o perfil e as regras da usuária
    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"],
        examples=None
    )
    user_prompt = triage_user_prompt.format(
        author=author, 
        to=to, 
        subject=subject, 
        email_thread=email_thread
    )

    # Invoca o LLM router com saída estruturada (retorna instância de Router)
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    if result.classification == "respond":
        print("📧 Classificação: RESPONDER - Este e-mail requer uma resposta")
        goto = "response_agent"
        # Injeta o e-mail completo como mensagem human para o response_agent processar
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Responda ao e-mail {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Classificação: IGNORAR - Este e-mail pode ser ignorado com segurança")
        update = None  # Nenhuma atualização de estado necessária
        goto = END
    elif result.classification == "notify":
        # Em um cenário real, isso poderia criar uma notificação ou registrar num log
        print("🔔 Classificação: NOTIFICAR - Este e-mail contém informações importantes")
        update = None
        goto = END
    else:
        raise ValueError(f"Classificação inválida: {result.classification}")

    # Command encapsula roteamento + atualização de estado num único objeto
    return Command(goto=goto, update=update)

## 14. Construindo e Compilando o Grafo

O grafo tem uma estrutura deliberadamente simples:

```
START → triage_router → (via Command) → response_agent | END
```

- `triage_router` usa `Command` para rotear, dispensando `add_conditional_edges`
- `response_agent` é o agente ReAct criado com `create_react_agent` — internamente é um sub-grafo completo (por isso `xray=True` na visualização)
- Não há `checkpointer` neste grafo: o estado não é persistido entre invocações (diferente da Aula 04). Para memória persistente, veja a Aula 08.

In [ ]:
email_agent = StateGraph(State)

# Adiciona o nó de triagem: classifica o e-mail e roteia via Command
email_agent = email_agent.add_node("triage_router", triage_router)

# Adiciona o agente de resposta (sub-grafo ReAct criado com create_react_agent)
email_agent = email_agent.add_node("response_agent", agent)

# O grafo sempre começa pelo nó de triagem
email_agent = email_agent.add_edge(START, "triage_router")

# Compila o grafo (sem checkpointer: sem persistência entre invocações)
email_agent = email_agent.compile()

## 15. Visualizando o Grafo

`xray=True` expande os sub-grafos internos (como o `response_agent`) na visualização, mostrando o ciclo LLM→ferramentas dentro do agente ReAct. Sem `xray=True`, `response_agent` apareceria como um único nó opaco.

In [ ]:
# xray=True expande os sub-grafos (ex.: response_agent) na visualização
display(Image(email_agent.get_graph(xray=True).draw_mermaid_png()))

## 16. Demonstrações — Spam e E-mail Legítimo

Testamos o sistema completo com dois casos polares:

### Caso 1: Spam (esperado: `ignore`)
Um e-mail de marketing com linguagem sensacionalista e oferta de desconto. O router deve reconhecer que não se enquadra em nenhuma das regras de `notify` ou `respond` e encerrar silenciosamente.

### Caso 2: E-mail legítimo da Alice (esperado: `respond`)
A mesma pergunta técnica sobre endpoints da API da seção 4. O router deve classificar como `respond` e acionar o `response_agent`, que usará `write_email` para redigir a resposta.

In [31]:
email_input = {
    "author": "Equipe de Marketing <marketing@amazingdeals.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "🔥 OFERTA EXCLUSIVA: Desconto por Tempo Limitado em Ferramentas para Desenvolvedores! 🔥",
    "email_thread": """Prezado(a) Desenvolvedor(a),

Não perca esta oportunidade INCRÍVEL! 

🚀 POR TEMPO LIMITADO, obtenha 80% DE DESCONTO em nosso Pacote Premium para Desenvolvedores! 

✨ RECURSOS:
- Preenchimento de código revolucionário com IA
- Ambiente de desenvolvimento baseado em nuvem
- Suporte ao cliente 24/7
- E muito mais!

💰 Preço Normal: R$ 999/mês
🎉 SEU PREÇO ESPECIAL: Apenas R$ 199/mês!

🕒 Corra! Esta oferta expira em:
APENAS 24 HORAS!

Clique aqui para resgatar seu desconto: https://amazingdeals.com/special-offer

Atenciosamente,
Equipe de Marketing
---
Para cancelar a inscrição, clique aqui
""",
}

In [ ]:
# Caso 1: e-mail de spam — esperamos classificação 'ignore'
response = email_agent.invoke({"email_input": email_input})

In [33]:
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "Dúvida rápida sobre a documentação da API",
    "email_thread": """Olá Sarah,

Eu estava revisando a documentação da API para o novo serviço de autenticação e notei que alguns endpoints parecem estar faltando nas especificações. Você poderia me ajudar a esclarecer se isso foi intencional ou se devemos atualizar a documentação?

Especificamente, estou procurando por:
- /auth/refresh
- /auth/validate

Obrigada!
Alice""",
}

In [ ]:
# Caso 2: e-mail legítimo de colega — esperamos classificação 'respond'
response = email_agent.invoke({"email_input": email_input})

In [ ]:
# Exibe todas as mensagens do ciclo: human → ai (tool_call) → tool → ai (resposta final)
for m in response["messages"]:
    m.pretty_print()